In [6]:
"""
Stage 2 — Dataset Preprocessing
=================================
Converts bounding box annotations (CCL'20 XML + Mendeley YOLO TXT)
into binary segmentation masks for U-Net training.

Why masks from bounding boxes?
  We don't have pixel-level annotations, so the bounding box region
  is used as a proxy for the infected area. This is called
  "weak supervision" — accepted practice in plant disease imaging.

Output structure:
  stage2_dataset/
    train/
      images/   ← resized original images (512×512 JPG)
      masks/    ← binary masks (512×512 PNG) — white=infected, black=background
    val/
      images/
      masks/
    test/
      images/
      masks/
    dataset_info.json  ← summary stats for your report

Run:
  python stage2_preprocess.py
  python stage2_preprocess.py --dry-run   # preview without writing files
"""

import json
import shutil
import random
import argparse
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np

# ══════════════════════════════════════════════════════════════════════════════
# ❶  CONFIGURE YOUR PATHS
# ══════════════════════════════════════════════════════════════════════════════

CCL20_IMAGES_DIR      = "CCL'20 dataset"        # ← change
CCL20_ANNOTATIONS_DIR = "CCL'20 dataset/train"       # ← change (.xml files)

YOLO_IMG_DIR          = "Lemon Leaf Diseases Dataset and Annotations_Mendeley/Lemon Leaf Diseases Dataset"       # ← change
YOLO_TXT_DIR          = "Lemon Leaf Diseases Dataset and Annotations_Mendeley/Lemon Leaf Diseases - Annotations"       # ← change

OUTPUT_DIR   = "stage2_dataset"
TARGET_SIZE  = 512          # resize all images + masks to this
SPLIT        = (0.70, 0.15, 0.15)
SEED         = 42
IMAGE_EXTS   = {".jpg", ".jpeg", ".png", ".bmp"}

# ══════════════════════════════════════════════════════════════════════════════
# ❷  LABEL MAP  (raw annotation name → unified class)
# ══════════════════════════════════════════════════════════════════════════════

LABEL_MAP = {
    # CCL'20 encoded labels
    "Disease-A":            "Anthracnose",
    "Disease-B":            "Melanose",
    "Disease-C":            "BlackSpot",
    # Common variants
    "Anthracnose":          "Anthracnose",
    "anthracnose":          "Anthracnose",
    "Canker":               "Canker",
    "canker":               "Canker",
    "Citrus_Canker":        "Canker",
    "Citrus Canker":        "Canker",
    "Greening":             "Greening",
    "HLB":                  "Greening",
    "Bacterial Blight":     "BacterialBlight",
    "BacterialBlight":      "BacterialBlight",
    "Curl Virus":           "CurlVirus",
    "CurlVirus":            "CurlVirus",
    "Deficiency Leaf":      "DeficiencyLeaf",
    "DeficiencyLeaf":       "DeficiencyLeaf",
    "Dry Leaf":             "DryLeaf",
    "DryLeaf":              "DryLeaf",
    "Sooty Mould":          "SootyMould",
    "SootyMould":           "SootyMould",
    "Spider Mites":         "SpiderMites",
    "SpiderMites":          "SpiderMites",
    "Healthy Leaf":         "Healthy",
    "Healthy":              "Healthy",
    "Melanose":             "Melanose",
    "BlackSpot":            "BlackSpot",
    "Black_spot":           "BlackSpot",
}

# ══════════════════════════════════════════════════════════════════════════════
# ❸  COLLECTORS
# ══════════════════════════════════════════════════════════════════════════════

def find_image(stem: str, search_dirs: list[Path]) -> Path | None:
    """Search for an image file by stem across multiple directories."""
    for d in search_dirs:
        for ext in IMAGE_EXTS:
            for candidate in d.rglob(stem + ext):
                if candidate.exists():
                    return candidate
    return None


def collect_ccl20(images_dir: str, ann_dir: str) -> list[dict]:
    """
    Parse Pascal VOC XML files from CCL'20.
    Each XML → one record with image path + list of bounding boxes.
    A single image may contain multiple disease lesions.
    """
    img_dir = Path(images_dir)
    ann_dir = Path(ann_dir)

    if not ann_dir.exists():
        print(f"  [MISSING] {ann_dir}")
        return []

    records = []
    skipped_labels = set()
    skipped_images = 0

    for xml_file in sorted(ann_dir.glob("*.xml")):
        try:
            tree = ET.parse(xml_file)
            root = tree.getroot()
        except ET.ParseError:
            print(f"  [WARN] Bad XML: {xml_file.name}")
            continue

        # Find matching image
        filename = root.findtext("filename", default="")
        img_path = img_dir / filename if filename else None
        if img_path is None or not img_path.exists():
            img_path = find_image(xml_file.stem, [img_dir])
        if img_path is None:
            skipped_images += 1
            continue

        # Read image dimensions from XML (faster than loading image)
        size_node = root.find("size")
        if size_node is not None:
            w = int(size_node.findtext("width",  "0") or "0")
            h = int(size_node.findtext("height", "0") or "0")
        else:
            w = h = 0

        # If dimensions missing, read from image
        if w == 0 or h == 0:
            img = cv2.imread(str(img_path))
            if img is None:
                skipped_images += 1
                continue
            h, w = img.shape[:2]

        # Parse bounding boxes
        boxes = []
        for obj in root.findall("object"):
            raw   = obj.findtext("name", "").strip()
            label = LABEL_MAP.get(raw)
            if label is None:
                skipped_labels.add(raw)
                continue
            bb = obj.find("bndbox")
            if bb is None:
                continue
            xmin = int(float(bb.findtext("xmin", "0")))
            ymin = int(float(bb.findtext("ymin", "0")))
            xmax = int(float(bb.findtext("xmax", "0")))
            ymax = int(float(bb.findtext("ymax", "0")))
            if xmax > xmin and ymax > ymin:
                boxes.append({
                    "label": label,
                    "xmin": xmin, "ymin": ymin,
                    "xmax": xmax, "ymax": ymax,
                })

        if boxes:
            records.append({
                "image_path": img_path,
                "width": w, "height": h,
                "boxes": boxes,
                "source": "CCL20_XML",
            })

    if skipped_labels:
        for sl in sorted(skipped_labels):
            print(f"  [SKIP label] '{sl}' not in LABEL_MAP")
    if skipped_images:
        print(f"  [SKIP] {skipped_images} images not found on disk")

    print(f"  CCL20 XML  → {len(records)} annotated images  "
          f"({sum(len(r['boxes']) for r in records)} total boxes)")
    return records


def collect_yolo(yolo_img_dir: str, yolo_txt_dir: str) -> list[dict]:
    """
    Parse YOLO TXT annotation files.
    Each line: class_id  cx  cy  bw  bh  (all 0–1 normalised)
    """
    txt_root = Path(yolo_txt_dir)
    img_root = Path(yolo_img_dir)
    if not txt_root.exists():
        print(f"  [MISSING] {txt_root}")
        return []

    classes = ["Anthracnose", "Bacterial Blight", "Citrus Canker", "Curl Virus", "Deficiency Leaf", "Dry Leaf", "Healthy Leaf", "Sooty Mould", "Spider Mites"]
    print(f"  YOLO classes: {classes}")

    records = []
    skipped_labels = set()
    skipped_images = 0

    for txt_file in sorted(txt_root.rglob("*.txt")):
        # Find matching image
        img_path = find_image(txt_file.stem, [img_root])
        if img_path is None:
            skipped_images += 1
            continue

        # Get image dimensions
        img = cv2.imread(str(img_path))
        if img is None:
            skipped_images += 1
            continue
        h, w = img.shape[:2]

        # Parse boxes
        boxes = []
        for line in txt_file.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            try:
                cls_id = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:5])
            except ValueError:
                continue

            raw   = classes[cls_id] if cls_id < len(classes) else str(cls_id)
            label = LABEL_MAP.get(raw)
            if label is None:
                skipped_labels.add(raw)
                continue

            # Convert normalised → pixel coords
            xmin = max(0, int((cx - bw / 2) * w))
            ymin = max(0, int((cy - bh / 2) * h))
            xmax = min(w, int((cx + bw / 2) * w))
            ymax = min(h, int((cy + bh / 2) * h))

            if xmax > xmin and ymax > ymin:
                boxes.append({
                    "label": label,
                    "xmin": xmin, "ymin": ymin,
                    "xmax": xmax, "ymax": ymax,
                })

        if boxes:
            records.append({
                "image_path": img_path,
                "width": w, "height": h,
                "boxes": boxes,
                "source": "Mendeley_YOLO",
            })

    if skipped_labels:
        for sl in sorted(skipped_labels):
            print(f"  [SKIP label] '{sl}' not in LABEL_MAP")
    if skipped_images:
        print(f"  [SKIP] {skipped_images} images not found on disk")

    print(f"  YOLO TXT   → {len(records)} annotated images  "
          f"({sum(len(r['boxes']) for r in records)} total boxes)")
    return records


# ══════════════════════════════════════════════════════════════════════════════
# ❹  MASK GENERATION
# ══════════════════════════════════════════════════════════════════════════════

def generate_mask(record: dict, target_size: int) -> tuple[np.ndarray, np.ndarray] | tuple[None, None]:
    """
    Loads the image, resizes to target_size × target_size.
    Draws filled white rectangles on a black canvas for each bounding box
    (scaled proportionally to the new size).

    Returns: (resized_image BGR, binary_mask uint8)
      mask pixel values: 0 = background, 255 = infected region
    """
    img = cv2.imread(str(record["image_path"]))
    if img is None:
        return None, None

    orig_h, orig_w = img.shape[:2]

    # Use recorded dimensions if image loaded at different size
    src_w = record["width"]  if record["width"]  > 0 else orig_w
    src_h = record["height"] if record["height"] > 0 else orig_h

    scale_x = target_size / src_w
    scale_y = target_size / src_h

    img_resized = cv2.resize(img, (target_size, target_size),
                             interpolation=cv2.INTER_LINEAR)
    mask = np.zeros((target_size, target_size), dtype=np.uint8)

    for box in record["boxes"]:
        x1 = max(0, int(box["xmin"] * scale_x))
        y1 = max(0, int(box["ymin"] * scale_y))
        x2 = min(target_size - 1, int(box["xmax"] * scale_x))
        y2 = min(target_size - 1, int(box["ymax"] * scale_y))
        if x2 > x1 and y2 > y1:
            mask[y1:y2, x1:x2] = 255

    return img_resized, mask


def compute_severity(mask: np.ndarray) -> dict:
    """
    Computes infected area percentage from mask and assigns severity label.
    Used for dataset_info.json — gives severity distribution stats.
    """
    infected_pct = (mask > 127).mean() * 100
    if infected_pct < 5:
        severity = "Healthy"
    elif infected_pct < 25:
        severity = "Mild"
    elif infected_pct < 50:
        severity = "Moderate"
    else:
        severity = "Severe"
    return {"infected_pct": round(float(infected_pct), 2), "severity": severity}


# ══════════════════════════════════════════════════════════════════════════════
# ❺  SPLIT + WRITE
# ══════════════════════════════════════════════════════════════════════════════

def write_split(records: list[dict], split_name: str,
                out_root: Path, target_size: int,
                dry_run: bool) -> list[dict]:
    """Write images and masks for one split. Returns per-record stats."""
    img_dir  = out_root / split_name / "images"
    mask_dir = out_root / split_name / "masks"
    if not dry_run:
        img_dir.mkdir(parents=True, exist_ok=True)
        mask_dir.mkdir(parents=True, exist_ok=True)

    written = 0
    skipped = 0
    stats   = []

    for i, rec in enumerate(records):
        if dry_run:
            written += 1
            continue

        img, mask = generate_mask(rec, target_size)
        if img is None:
            skipped += 1
            continue

        stem = f"{split_name}_{i:05d}"
        cv2.imwrite(str(img_dir  / f"{stem}.jpg"), img,
                    [cv2.IMWRITE_JPEG_QUALITY, 95])
        cv2.imwrite(str(mask_dir / f"{stem}.png"), mask)

        sev = compute_severity(mask)
        stats.append({
            "file":         stem,
            "source":       rec["source"],
            "labels":       list({b["label"] for b in rec["boxes"]}),
            "num_boxes":    len(rec["boxes"]),
            "infected_pct": sev["infected_pct"],
            "severity":     sev["severity"],
        })
        written += 1

    print(f"  {split_name:5s}: {written} pairs written  "
          f"({skipped} skipped)" + (" [DRY RUN]" if dry_run else ""))
    return stats


# ══════════════════════════════════════════════════════════════════════════════
# ❻  REPORT
# ══════════════════════════════════════════════════════════════════════════════

def print_report(all_stats: dict, total: int):
    from collections import Counter

    print("\n" + "═" * 65)
    print("  STAGE 2 DATASET REPORT")
    print("═" * 65)

    for split_name, stats in all_stats.items():
        if not stats:
            continue
        sev_counts = Counter(s["severity"] for s in stats)
        src_counts = Counter(s["source"]   for s in stats)
        avg_pct    = sum(s["infected_pct"] for s in stats) / len(stats)

        print(f"\n  {split_name.upper()}  ({len(stats)} images)")
        print(f"  {'Severity':<12}  Count")
        for sev in ["Healthy", "Mild", "Moderate", "Severe"]:
            n   = sev_counts.get(sev, 0)
            bar = "▓" * (n * 20 // max(len(stats), 1))
            print(f"    {sev:<10}  {n:>5}  {bar}")
        print(f"  Avg infected area : {avg_pct:.1f}%")
        print(f"  Sources           : {dict(src_counts)}")

    print(f"\n  TOTAL pairs: {total}")
    print("═" * 65)


# ══════════════════════════════════════════════════════════════════════════════
# ❼  MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main(dry_run: bool = False):
    random.seed(SEED)
    print("=" * 65)
    print("  Stage 2 — Dataset Preprocessing")
    print("=" * 65)
    if dry_run:
        print("  ▶️  DRY RUN — no files will be written\n")

    # ── Collect ───────────────────────────────────────────────────────────────
    print("\nCollecting annotations...")
    records = []
    records += collect_ccl20(CCL20_IMAGES_DIR, CCL20_ANNOTATIONS_DIR)
    records += collect_yolo(YOLO_IMG_DIR, YOLO_TXT_DIR)

    # Deduplicate by resolved absolute path
    seen, unique = set(), []
    for r in records:
        p = r["image_path"].resolve()
        if p not in seen:
            seen.add(p)
            unique.append(r)

    print(f"\n  Total unique annotated images : {len(unique)}")
    print(f"  Total bounding boxes          : "
          f"{sum(len(r['boxes']) for r in unique)}")

    if not unique:
        print("\n[ERROR] No annotated images found. Check your paths above.")
        return

    # ── Split ─────────────────────────────────────────────────────────────────
    random.shuffle(unique)
    n  = len(unique)
    t1 = int(n * SPLIT[0])
    t2 = t1 + int(n * SPLIT[1])
    splits = {
        "train": unique[:t1],
        "val":   unique[t1:t2],
        "test":  unique[t2:],
    }

    print(f"\n  Split: train={len(splits['train'])}  "
          f"val={len(splits['val'])}  test={len(splits['test'])}")

    # ── Write ─────────────────────────────────────────────────────────────────
    out_root = Path(OUTPUT_DIR)
    if not dry_run and out_root.exists():
        shutil.rmtree(out_root)

    print(f"\nGenerating masks → {OUTPUT_DIR}/")
    print(f"  Target image size : {TARGET_SIZE}×{TARGET_SIZE}")

    all_stats: dict[str, list] = {}
    total_written = 0
    for split_name, recs in splits.items():
        stats = write_split(recs, split_name, out_root, TARGET_SIZE, dry_run)
        all_stats[split_name] = stats
        total_written += len(stats)

    # ── Save info JSON ────────────────────────────────────────────────────────
    if not dry_run:
        info = {
            "target_size": TARGET_SIZE,
            "split_ratio": list(SPLIT),
            "total":       total_written,
            "splits":      {k: len(v) for k, v in all_stats.items()},
            "records":     {k: v for k, v in all_stats.items()},
        }
        info_path = out_root / "dataset_info.json"
        with open(info_path, "w") as f:
            json.dump(info, f, indent=2)
        print(f"\n  ✓ Dataset info saved → {info_path}")

    # ── Report ────────────────────────────────────────────────────────────────
    if not dry_run:
        print_report(all_stats, total_written)
        print(f"\n✓ Stage 2 dataset ready at ./{OUTPUT_DIR}/")
        print("  Structure:")
        print("    stage2_dataset/")
        print("      train/images/  ← 512×512 JPG")
        print("      train/masks/   ← 512×512 PNG  (white=infected)")
        print("      val/images/  val/masks/")
        print("      test/images/ test/masks/")
        print("      dataset_info.json")
    else:
        print(f"\n  Dry run complete. Would write {total_written} image+mask pairs.")
        print("  Run without --dry-run to generate the actual files.")


main(dry_run=False)

  Stage 2 — Dataset Preprocessing

  CCL20 XML  → 1817 annotated images  (8331 total boxes)
  YOLO classes: ['Anthracnose', 'Bacterial Blight', 'Citrus Canker', 'Curl Virus', 'Deficiency Leaf', 'Dry Leaf', 'Healthy Leaf', 'Sooty Mould', 'Spider Mites']
  YOLO TXT   → 900 annotated images  (901 total boxes)

  Total unique annotated images : 2717
  Total bounding boxes          : 9232

  Split: train=1901  val=407  test=409

Generating masks → stage2_dataset/
  Target image size : 512×512
  train: 1901 pairs written  (0 skipped)
  val  : 407 pairs written  (0 skipped)
  test : 409 pairs written  (0 skipped)

  ✓ Dataset info saved → stage2_dataset/dataset_info.json

═════════════════════════════════════════════════════════════════
  STAGE 2 DATASET REPORT
═════════════════════════════════════════════════════════════════

  TRAIN  (1901 images)
  Severity      Count
    Healthy       488  ▓▓▓▓▓
    Mild          365  ▓▓▓
    Moderate      361  ▓▓▓
    Severe        687  ▓▓▓▓▓▓▓
  Avg inf